# `calibrate(method="brightness", ...)` — `brightness`

Converter radiância térmica em temperatura de brilho TOA.

**Cadeia deste exemplo:** Radiância B10 ou B11 + MTL + QA → BT em K.

Este notebook executa uma etapa sobre arquivos já preparados. **Se você tem apenas os arquivos originais Level-1, prepare as entradas indicadas na tabela antes de executar este método.**

Execute apenas os **quatro blocos de código**, de cima para baixo. Os blocos ao final, em “Consulta”, são alternativas para leitura, e não uma continuação obrigatória.


Esta conversao pertence a familia `calibration`. Use `calibrate(data, method="brightness", ...)`. A funcao direta `brightness(...)` continua disponivel para compatibilidade.


## Antes de executar: separe estes arquivos

| Campo a preencher no passo 1 | Arquivo que você deve escolher | Unidade / tipo | Como obter se ainda falta |
| --- | --- | --- | --- |
| `radiancia_path` | Radiância térmica B10; por exemplo, `radiancia_B10.tif` | Radiância TOA; W m⁻² sr⁻¹ µm⁻¹ | [Radiância TOA](toa_radiance.ipynb) com `banda=10` |
| `mtl_path` | Metadados de calibração; por exemplo, `PRODUTO_MTL.txt` | Arquivo texto MTL | Arquivo original da mesma cena Level-1 |
| `qa_pixel_path` | Qualidade dos pixels; por exemplo, `PRODUTO_QA_PIXEL.TIF` | Bits inteiros, sem calibração | Arquivo original QA_PIXEL da mesma cena e na mesma grade |
| `qa_radsat_path` | Saturação/qualidade geométrica; por exemplo, `PRODUTO_QA_RADSAT.TIF` | Bits inteiros, sem calibração | Arquivo original QA_RADSAT da mesma cena e na mesma grade |

Todos os GeoTIFFs devem representar a mesma cena e ter dimensões, CRS e transformação idênticos. Para recortes, use a mesma janela nas QA. As unidades são verificadas pelo usuário: a leitura do arquivo não comprova sua grandeza física.

**Configuração desta execução:** Comece com radiância B10 e `banda=10`. Para BT11, troque o arquivo por radiância B11 e use `banda=11`.

**Ambiente:** instale `python -m pip install -e ".[examples]"` na pasta da biblioteca e selecione o kernel desse ambiente no VS Code/Jupyter.


## Passo 1 — Preencher caminhos e escolher a saída

Substitua cada `CAMINHO_...` pelo caminho completo do seu arquivo, mantendo `Path(...)`. No Windows, escreva, por exemplo, `Path(r"C:\dados\radiancia_B4.tif")`. Preencha todos os campos da tabela, inclusive as duas QA.

Comece com radiância B10 e `banda=10`. Para BT11, troque o arquivo por radiância B11 e use `banda=11`.

Ajuste `saida` para a pasta desejada. Caminhos relativos partem da pasta de trabalho do kernel; a célula imprime o caminho absoluto. Em nova execução com o mesmo destino, o GeoTIFF é substituído.


In [ ]:
from pathlib import Path
import numpy as np
import rasterio

# EDITE OS CAMINHOS E AS CONFIGURAÇÕES ABAIXO; mantenha o método deste notebook.
from pylstemp import quality_mask
from pylstemp import calibrate, read_mtl

radiancia_path = Path("CAMINHO_DA_RADIANCIA_TERMICA.tif")

banda = 10
mtl_path = Path("CAMINHO_DO_MTL.txt")
qa_pixel_path = Path("CAMINHO_DO_QA_PIXEL.tif")
qa_radsat_path = Path("CAMINHO_DO_QA_RADSAT.tif")
saida = Path("resultados/brightness") / f"B{banda}"

# Confira os arquivos antes de passar ao passo 2.
entradas_obrigatorias = {
    "radiancia_path": radiancia_path,
    "mtl_path": mtl_path,
    "qa_pixel_path": qa_pixel_path,
    "qa_radsat_path": qa_radsat_path,
}
pendentes = [f"{nome}: {caminho}" for nome, caminho in entradas_obrigatorias.items()
             if not caminho.is_file()]
if pendentes:
    raise FileNotFoundError("Preencha os caminhos de entrada com arquivos existentes:\n"
                            + "\n".join(pendentes))
print("Arquivos encontrados. Próximo passo: ler os dados e aplicar QA.")
print("Pasta de saída:", saida.resolve())


## Passo 2 — Ler as entradas, conferir a grade e aplicar QA

Execute esta célula depois de confirmar os caminhos. Ela carrega os dados, confere seu alinhamento, lê QA_PIXEL/QA_RADSAT com `quality_mask()` e combina QA com nodata em `mascara`. `True` indica pixel excluído. A leitura não recalibra dados prontos.

Água é preservada na política QA deste exemplo (`exclude_water=False`). A célula imprime as dimensões e as contagens de exclusão.


In [ ]:
arquivos = {
    "radiancia": radiancia_path,
}
dados = {}
grade = None
for nome, caminho in arquivos.items():
    with rasterio.open(caminho) as src:
        grade_atual = (src.shape, src.crs, src.transform)
        if grade is None:
            grade = grade_atual
            perfil = src.profile.copy()
        elif grade_atual != grade:
            raise ValueError(f"A imagem {nome} não está na mesma grade das outras entradas.")
        dados[nome] = src.read(1, masked=True).astype(float).filled(np.nan)
radiancia = dados["radiancia"]
mascara = ~np.isfinite(radiancia)
metadata = read_mtl(mtl_path)
# Ler QA na grade da primeira imagem e combinar com nodata antes do método.
mascara_qa = quality_mask(
    qa_pixel_path, qa_radsat_path,
    reference_path=next(iter(arquivos.values())),
    exclude_water=False,
)
mascara |= mascara_qa

print("Dimensões:", mascara.shape)
print("Pixels excluídos por QA:", int(mascara_qa.sum()))
print("Pixels excluídos por QA ou nodata:", int(mascara.sum()), "de", mascara.size)


## Passo 3 — Calcular o produto

Execute a chamada abaixo. As variáveis são matrizes carregadas no passo 2; os caminhos de arquivo foram usados no passo 1. `mask=mascara` aplica a seleção QA/nodata ao cálculo.


In [ ]:
resultado = calibrate(
    data=radiancia,
    method="brightness",
    band=banda,
    metadata=metadata,
    input_quantity="radiance",
    mask=mascara,
)
imagem = resultado

## Passo 4 — Salvar e usar a saída

Execute esta célula para gravar o GeoTIFF em `saida`, preservando a grade. Confira o caminho absoluto e a quantidade de pixels válidos impressos. Pixels inválidos são gravados como NaN.

**Próximo uso:** BT10/BT11 em K alimentam PWV e os métodos de temperatura. Esta saída é temperatura de brilho, não LST.


In [ ]:
saida.mkdir(parents=True, exist_ok=True)
perfil.update(driver="GTiff", dtype="float32", count=1, nodata=np.nan, compress="deflate")
with rasterio.open(saida / "brightness.tif", "w", **perfil) as dst:
    dst.write(np.asarray(imagem, dtype="float32"), 1)
    dst.update_tags(units='K', method='brightness')
print("Pixels válidos:", int(np.isfinite(imagem).sum()), "de", imagem.size)
print("Saída:", saida.resolve())

## Como conferir a execução

Abra o GeoTIFF salvo em um SIG e confira a posição, a unidade e a banda informadas acima. Compare a contagem de pixels válidos com a área disponível: QA pode excluir pixels, e os critérios do próprio método podem produzir lacunas adicionais.

| Se acontecer | O que conferir |
| --- | --- |
| `FileNotFoundError` no passo 1 | Preencha os caminhos listados no erro, inclusive MTL quando solicitado e ambas as QA. |
| Erro de grade no passo 2 | Prepare arquivos com a mesma projeção, transformação, dimensões e recorte. Não redimensione as QA por interpolação contínua. |
| Erro de domínio no passo 3 | Confira a grandeza/unidade e os requisitos do método. |
| Saída com NaN | Confira as contagens QA/nodata e o suporte do método; NaN não deve ser substituído por temperatura inventada. |

[Voltar ao índice e à ordem de execução](README.md).


## Consulta — opções e detalhes do método

O roteiro executável termina acima. Leia os tópicos abaixo para adaptar a entrada ou consultar parâmetros; não execute todas as alternativas em sequência sobre a mesma saída.


### Entradas da função e opções

### Entradas do caminho principal

| Entrada da função | O que fornecer | Unidade / domínio |
|---|---|---|
| `data` | Imagem de radiância TOA da banda térmica escolhida | W m⁻² sr⁻¹ µm⁻¹ |

**Além da imagem:** número da banda térmica (10 ou 11) e `metadata`, dicionário lido do MTL correspondente. O padrão de entrada é radiância pronta. A alternativa DN faz a calibração internamente e não deve receber uma radiância já convertida.

Todas as imagens usadas juntas devem ter a mesma grade: projeção, resolução, posição dos pixels e dimensões. As variáveis da chamada são arrays de pixels; não são caminhos de arquivo.

### O que é opcional

- **`mask`:** máscara booleana alinhada: True exclui o pixel e False permite avaliar. Opcional na API; neste exemplo é preenchida no passo 2 com QA e nodata. Não passar QA_PIXEL bruta.

### Alternativas de uso

Escolha a alternativa que corresponde aos dados disponíveis; não execute todas as chamadas em sequência sobre a mesma saída.

### 5.1. Entrar com DN e MTL

Esta chamada faz DN→radiância→BT dentro da função. Não fornecer radiância pronta com input_quantity='dn'.

```python
imagem = calibrate(
    data=dn,
    method="brightness",
    band=banda,
    input_quantity="dn",
    metadata=metadata,
    mask=mascara,
)
```
### 5.2. Fallback sem MTL

Se não houver MTL, é possível identificar o sensor e usar as constantes de fallback. Isso não verifica a calibração real da cena. Não fornecer metadata simultaneamente com overrides de ganho/offset; em radiância pronta, rad_gain/rad_bias são rejeitados.

```python
imagem = calibrate(
    data=radiancia,
    method="brightness",
    band=banda,
    sensor="landsat_8",
    input_quantity="radiance",
    mask=mascara,
)
```

### Referência completa dos parâmetros

### Caminho principal

A chamada completa e a tabela abaixo seguem a mesma ordem.

```python
resultado = calibrate(
    data=radiancia,
    method="brightness",
    band=banda,
    metadata=metadata,
    input_quantity="radiance",
    mask=mascara,
)
```

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `data` | Obrigatório | `Sem padrão` | W m⁻² sr⁻¹ µm⁻¹ ou DN | Imagem térmica TOA em radiância por padrão; declarar input_quantity="dn" para DN. Mesma cena/banda do MTL. |
| `method` | Obrigatorio | Sem padrao | Seletor | `"brightness"` neste exemplo. |
| `band` | Obrigatório | `Sem padrão` | Número da banda | 10 ou 11; aliases b10/b11 e band_10/band_11 também aceitos. Usar o MTL correspondente à banda térmica. |
| `metadata` | Obrigatório na rota deste exemplo | `None` | Dicionário MTL | MTL da mesma cena; fornece K1/K2 e, para DN, ganho/offset. O arquivo é lido por LandsatScene/read_mtl. |
| `input_quantity` | Opcional | `"radiance"` | Seletor | "radiance" ou "dn"; declara a entrada. DN é convertido com os coeficientes da mesma banda no MTL. |
| `mask` | Opcional na API; fornecido neste exemplo | `None` | Booleano | Máscara de exclusão alinhada: True rejeita, False permite avaliar. O passo 2 lê ambas as QA e combina seus bits com nodata. Água segue a escolha exclude_water explícita neste notebook. Não fornecer QA bruta; a função pode acrescentar rejeições por domínio. |

### Parâmetros das alternativas

As entradas substituídas e as combinações permitidas são explicadas na consulta de alternativas. Os demais parâmetros permanecem os da tabela principal.

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `sensor` | Condicional: fallback sem MTL | `None → landsat_8` | Seletor | "landsat_8" ou "landsat_9"; usa constantes térmicas de fallback. Preferir metadata da própria cena; omitir no fluxo com MTL. |
| `rad_gain` | Opcional no fallback DN; recomendado informar | `None → constante do sensor` | Radiância por DN | Ganho radiométrico da banda, obtido na própria cena; não reutilizar coeficientes de outra cena. |
| `rad_bias` | Opcional no fallback DN; recomendado informar | `None → constante do sensor` | W m⁻² sr⁻¹ µm⁻¹ | Offset radiométrico da mesma banda e cena. |

Os valores escolhidos no exemplo podem diferir do padrão da API; a coluna de padrão descreve a função. As máscaras e as validações de domínio podem produzir NaN, sem preenchimento automático.

### Aplicabilidade e limites

O sensor/MTL devem ser da própria imagem. DN, radiância, BT e LST são grandezas diferentes. A transformação de Planck não faz correção de emissividade ou atmosfera.

Execução e consistência numérica não demonstram acurácia independente em campo. A parametrização deve ser adequada à banda, sensor, superfície e condições da aplicação.

[Artigo ou documentacao original](https://www.usgs.gov/landsat-missions/using-usgs-landsat-level-1-data-product).

Veja a [ficha metodologica](../docs/method-cards.md) para equacao, coeficientes, dominio e adaptacoes. Execucao numericamente correta nao estabelece acuracia de campo. [Voltar ao indice](README.md).